In [1]:
!git clone https://github.com/Kaihua-Chen/diffusion-vas
%cd diffusion-vas

%mkdir checkpoints
%cd checkpoints
!git lfs install
!git clone https://huggingface.co/kaihuac/diffusion-vas-amodal-segmentation
!git clone https://huggingface.co/kaihuac/diffusion-vas-content-completion

!wget -O depth_anything_v2_vitl.pth https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth?download=true
%cd ../..


Cloning into 'diffusion-vas'...
remote: Enumerating objects: 462, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 462 (delta 29), reused 28 (delta 28), pack-reused 427 (from 1)
Receiving objects: 100% (462/462), 102.79 MiB | 56.62 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/diffusion-vas
/content/diffusion-vas/checkpoints
Updated git hooks.
Git LFS initialized.
Cloning into 'diffusion-vas-amodal-segmentation'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 48 (delta 11), reused 0 (delta 0), pack-reused 4 (from 1)
Unpacking objects: 100% (48/48), 28.43 KiB | 2.58 MiB/s, done.
Filtering content: 100% (3/3), 3.03 GiB | 42.04 MiB/s, done.
Encountered 1 file(s) that may not have been copied correctly on Windows:
	unet/diffusion_pytorch_model.safetensors

See: `git lfs help smudge` for more details.
Cloning int

In [7]:
!tar -xvf ff5da6d6ecae486bb294aeaf5ee8f8a1.tar.gz

Streaming output truncated to the last 5000 lines.
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00019.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00000.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00017.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00010.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00014.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00013.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00005.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00019.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00009.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00007.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00001.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00021.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00000.

In [8]:
from pathlib import Path
import shutil
import numpy as np
from PIL import Image

def copy_demo_format(
    root: Path,
    camera_index: int = 0,
    object_id: int = 1,
    target_root: Path = None
) -> Path:
    cam_folder = root / f"camera_{camera_index:04d}"

    if not cam_folder.exists():
        raise FileNotFoundError(f"Camera folder {cam_folder} not found.")

    if target_root is None:
        target_root = root

    # Create target structure
    new_folder = target_root / f"copy_cam_{camera_index}_obj_{object_id}"
    rgba_target = new_folder / "rgbs"
    seg_target = new_folder / "masks"
    rgba_target.mkdir(parents=True, exist_ok=True)
    seg_target.mkdir(parents=True, exist_ok=True)

    # Collect and sort source files
    rgba_files = sorted(cam_folder.glob("rgba_*.png"))
    seg_files = sorted(cam_folder.glob("segmentation_*.png"))

    assert len(rgba_files) == len(seg_files), "Mismatch between RGB and segmentation files."

    limit = 25
    count = min(limit, len(rgba_files))

    for i in range(count):
        # Copy RGBA image
        rgba_dest = rgba_target / f"rgba_{i:05d}.png"
        shutil.copy(rgba_files[i], rgba_dest)

        # Process and save mask for the correct object ID
        seg_img = np.array(Image.open(seg_files[i]))
        binary_mask = (seg_img == object_id).astype(np.uint8) * 255
        mask_img = Image.fromarray(binary_mask)
        mask_dest = seg_target / f"segmentation_{i:05d}.png"
        mask_img.save(mask_dest)

    # Pad RGBA and masks if fewer than 25 frames
    if count < limit:
        last_rgba = rgba_files[count - 1]
        last_seg = np.array(Image.open(seg_files[count - 1]))
        last_mask = (last_seg == object_id).astype(np.uint8) * 255
        last_mask_img = Image.fromarray(last_mask)

        for i in range(count, limit):
            shutil.copy(last_rgba, rgba_target / f"rgba_{i:05d}.png")
            last_mask_img.save(seg_target / f"segmentation_{i:05d}.png")

    return new_folder


In [9]:
from pathlib import Path

root_path = Path("/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000")
target_path = Path("/content/diffusion-vas/demo_data")

object_folders = sorted(root_path.glob("obj_*"))

for folder in object_folders:
    object_id = int(folder.name.split("_")[1])
    print(f"Setting up object {object_id:02d}...")

    copy_demo_format(
        root=Path("/content/ff5da6d6ecae486bb294aeaf5ee8f8a1"),
        camera_index=0,
        object_id=object_id,
        target_root=target_path
    )

    seq_name = f"copy_cam_0_obj_{object_id}"
    !cd /content/diffusion-vas && python demo.py --seq_name "{seq_name}"


Setting up object 01...
2025-07-24 21:20:07.879624: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-24 21:20:07.895375: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753392007.916041    8544 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753392007.922482    8544 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-24 21:20:07.943651: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to

In [10]:
import os
from PIL import Image
import glob

os.makedirs("/content/gt_gifs", exist_ok=True)

cam_path = "/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000"
object_dirs = sorted(glob.glob(f"{cam_path}/obj_*"))

for folder in object_dirs:
    object_id = int(folder.split("_")[-1])
    frame_paths = sorted(glob.glob(f"{folder}/rgba_*.png"))
    save_path = f"/content/gt_gifs/obj_{object_id:02d}.gif"

    if not frame_paths:
        print(f"No GT frames for object {object_id}")
        continue

    frames = [Image.open(p).convert("RGB").resize((256, 256)) for p in frame_paths]
    frames[0].save(save_path, save_all=True, append_images=frames[1:], duration=100, loop=0)
    print(f"Saved GT GIF: {save_path}")


Saved GT GIF: /content/gt_gifs/obj_01.gif
Saved GT GIF: /content/gt_gifs/obj_02.gif
Saved GT GIF: /content/gt_gifs/obj_03.gif
Saved GT GIF: /content/gt_gifs/obj_04.gif
Saved GT GIF: /content/gt_gifs/obj_05.gif
Saved GT GIF: /content/gt_gifs/obj_06.gif
Saved GT GIF: /content/gt_gifs/obj_07.gif
Saved GT GIF: /content/gt_gifs/obj_08.gif
Saved GT GIF: /content/gt_gifs/obj_09.gif
Saved GT GIF: /content/gt_gifs/obj_10.gif
Saved GT GIF: /content/gt_gifs/obj_11.gif
Saved GT GIF: /content/gt_gifs/obj_12.gif
Saved GT GIF: /content/gt_gifs/obj_13.gif
Saved GT GIF: /content/gt_gifs/obj_14.gif
Saved GT GIF: /content/gt_gifs/obj_15.gif
Saved GT GIF: /content/gt_gifs/obj_16.gif
Saved GT GIF: /content/gt_gifs/obj_17.gif
Saved GT GIF: /content/gt_gifs/obj_18.gif
Saved GT GIF: /content/gt_gifs/obj_19.gif


In [16]:
# Re-imports after code reset
from pathlib import Path
import imageio.v2 as imageio
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from PIL import Image
import re
import pandas as pd

# Initialize metric containers
all_psnr, all_ssim, all_ace, all_iou = [], [], [], []

# Find all predicted outputs matching the naming pattern
pred_base_path = Path("/content/diffusion-vas/outputs")
gt_base_path = Path("/content/gt_gifs")
pred_dirs = sorted(pred_base_path.glob("copy_cam_0_obj_*"))

# Extract object IDs from the folder names
for pred_dir in pred_dirs:
    match = re.search(r"obj_(\d+)", pred_dir.name)
    if not match:
        continue
    object_id = int(match.group(1))

    pred_path = pred_dir / "pred_amodal_rgb.gif"
    gt_path = gt_base_path / f"obj_{object_id:02d}.gif"

    if not pred_path.exists() or not gt_path.exists():
        print(f"Skipping obj_{object_id:02d} — missing predicted or ground truth GIF")
        continue

    pred_frames = imageio.mimread(str(pred_path))
    gt_frames = imageio.mimread(str(gt_path))

    target_size = (256, 256)
    min_len = min(len(pred_frames), len(gt_frames))

    pred_np = np.stack([
        np.array(Image.fromarray(f).resize(target_size).convert("RGB")).astype(np.float32) / 255.0
        for f in pred_frames[:min_len]
    ])
    gt_np = np.stack([
        np.array(Image.fromarray(f).resize(target_size).convert("RGB")).astype(np.float32) / 255.0
        for f in gt_frames[:min_len]
    ])

    def normalize_bg_white(x):
        return np.where((x < 0.05).all(axis=-1, keepdims=True), 1.0, x)

    pred_np = np.stack([normalize_bg_white(f) for f in pred_np])
    gt_np = np.stack([normalize_bg_white(f) for f in gt_np])

    for gt, pr in zip(gt_np, pred_np):
        all_psnr.append(psnr(gt, pr, data_range=1.0))
        all_ssim.append(ssim(gt, pr, channel_axis=2, data_range=1.0))

        # ACE over object pixels only
        bg_mask = ((gt > 0.95) & (pr > 0.95)).all(axis=-1)
        obj_mask = ~bg_mask
        if obj_mask.any():
            ace = np.mean(np.abs(gt[obj_mask] - pr[obj_mask]))
            all_ace.append(ace)

        # IoU from derived binary masks
        gt_mask = (gt < 0.95).any(axis=-1)
        pr_mask = (pr < 0.95).any(axis=-1)

        intersection = np.logical_and(gt_mask, pr_mask).sum()
        union = np.logical_or(gt_mask, pr_mask).sum()
        iou = intersection / union if union > 0 else 0
        all_iou.append(iou)

# Final averaged metrics
results = {
    "Average PSNR": np.mean(all_psnr),
    "Average SSIM": np.mean(all_ssim),
    "ACE (Object Pixels Only, White BG)": np.mean(all_ace),
    "IoU (from RGB-derived masks)": np.mean(all_iou)
}

print("Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

Evaluation Results:
Average PSNR: 26.6697
Average SSIM: 0.9732
ACE (Object Pixels Only, White BG): 0.2522
IoU (from RGB-derived masks): 0.7382


In [18]:
from PIL import Image
import numpy as np
import glob
import os

os.makedirs("/content/gt_mask_gifs", exist_ok=True)

for folder in object_dirs:
    seg_folder = f"/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000/obj_{object_id:04d}"
    save_path = f"/content/gt_mask_gifs/obj_{object_id:02d}_mask.gif"
    mask_paths = sorted(glob.glob(f"{seg_folder}/segmentation_*.png"))

    if not mask_paths:
        print(f"⚠️ No mask images for obj_{object_id}")
        continue

    frames = []
    for path in mask_paths:
        seg = np.array(Image.open(path))
        binary_mask = (seg > 0).astype(np.uint8) * 255  # fix here!
        frames.append(Image.fromarray(binary_mask).resize((256, 256)))

    frames[0].save(save_path, save_all=True, append_images=frames[1:], duration=100, loop=0)
    print(f"✅ Saved GT mask GIF: {save_path}")


✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.gif
✅ Saved GT mask GIF: /content/gt_mask_gifs/obj_10_mask.g

In [19]:
from PIL import Image
import numpy as np
import os

all_ious = []

for object_id in range(1, 11):
    gt_path = f"/content/gt_mask_gifs/obj_{object_id:02d}_mask.gif"
    pred_path = f"/content/diffusion-vas/outputs/copy_cam_0_obj_{object_id}/pred_amodal_masks.gif"

    if not os.path.exists(gt_path) or not os.path.exists(pred_path):
        print(f"Missing mask GIFs for obj_{object_id}")
        continue

    # Load and binarize frames
    gt_gif = Image.open(gt_path)
    pred_gif = Image.open(pred_path)

    gt_frames, pred_frames = [], []

    try:
        while True:
            gt_frame = gt_gif.copy().convert("L").resize((256, 256))
            pred_frame = pred_gif.copy().convert("L").resize((256, 256))

            gt_frames.append(np.array(gt_frame) > 127)
            pred_frames.append(np.array(pred_frame) > 127)

            gt_gif.seek(gt_gif.tell() + 1)
            pred_gif.seek(pred_gif.tell() + 1)
    except EOFError:
        pass

    min_len = min(len(gt_frames), len(pred_frames))
    ious = []

    for i in range(min_len):
        gt_mask = gt_frames[i]
        pred_mask = pred_frames[i]

        intersection = np.logical_and(gt_mask, pred_mask).sum()
        union = np.logical_or(gt_mask, pred_mask).sum()

        if union > 0:
            ious.append(intersection / union)

    if ious:
        mean_iou = np.mean(ious)
        print(f"Obj {object_id:02d} - Mean IoU: {mean_iou:.4f}")
        all_ious.extend(ious)

# Final mean IoU
overall_iou = np.mean(all_ious)
print(f"\nOverall Mean IoU (GT vs Pred Mask): {overall_iou:.4f}")


Obj 01 - Mean IoU: 0.9812
Obj 03 - Mean IoU: 0.9764
Obj 04 - Mean IoU: 0.9973
Obj 05 - Mean IoU: 0.5795
Obj 06 - Mean IoU: 0.2344
Obj 07 - Mean IoU: 0.5592
Obj 08 - Mean IoU: 0.7358
Obj 09 - Mean IoU: 0.7603
Obj 10 - Mean IoU: 0.9642

Overall Mean IoU (GT vs Pred Mask): 0.7352
